# Анализ сжатия данных методом Tensor Train decomposition

Этот ноутбук предназначен для:
- Сжатия данных концентрации CO с помощью Tensor Train (TT) разложения
- Анализа качества реконструкции
- Сравнения с PCA и автоэнкодером
- Подбора оптимальных TT-рангов

In [ ]:
cd ..

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import sys
from pathlib import Path

sys.path.insert(0, 'src')

from src.data import load_data, normalize_data, ConcentrationFieldDataset
from src.tensor_train import TTCompressor, analyze_tt_components
from src.pca import PCACompressor
from src.models import Conv3DAutoencoder
from src.train import load_model

from sklearn.model_selection import train_test_split

%matplotlib inline

## Загрузка данных

**Важно:** Используем те же параметры разбиения, что и при обучении автоэнкодера (random_state=42, val_split=0.10)

In [ ]:
# Список всех файлов (как в config.yaml)
ALL_FILES = [
    '../OUTPUT/Nsk_2024-07-01_00_00_to_2024-07-02_00_00/all_sources/uvwmn_d03_2024-07-01_00:00:00_n33_chem_13.nc',
    '../OUTPUT/Nsk_2024-07-13_00_00_to_2024-07-14_00_00/all_sources/uvwmn_d03_2024-07-13_00:00:00_n33_chem_13.nc',
    '../OUTPUT/Nsk_2024-07-28_00_00_to_2024-07-29_00_00/all_sources/uvwmn_d03_2024-07-28_00:00:00_n33_chem_13.nc',
    '../OUTPUT/Nsk_2024-08-05_00_00_to_2024-08-06_00_00/all_sources/uvwmn_d03_2024-08-05_00:00:00_n33_chem_13.nc',
    '../OUTPUT/Nsk_2024-08-26_00_00_to_2024-08-27_00_00/all_sources/uvwmn_d03_2024-08-26_00:00:00_n33_chem_13.nc',
    '../OUTPUT/Nsk_2024-09-05_00_00_to_2024-09-06_00_00/all_sources/uvwmn_d03_2024-09-05_00:00:00_n33_chem_13.nc',
    '../OUTPUT/Nsk_2024-09-12_00_00_to_2024-09-13_00_00/all_sources/uvwmn_d03_2024-09-12_00:00:00_n33_chem_13.nc',
    '../OUTPUT/Nsk_2025-04-29_00_00_to_2025-04-30_00_00/all_sources/uvwmn_d03_2025-04-29_00:00:00_n33_chem_13.nc',
    '../OUTPUT/Nsk_2025-05-09_00_00_to_2025-05-10_00_00/all_sources/uvwmn_d03_2025-05-09_00:00:00_n33_chem_13.nc',
    '../OUTPUT/Nsk_2025-05-15_00_00_to_2025-05-16_00_00/all_sources/uvwmn_d03_2025-05-15_00:00:00_n33_chem_13.nc',
    '../OUTPUT/Nsk_2025-06-18_00_00_to_2025-06-19_00_00/all_sources/uvwmn_d03_2025-06-18_00:00:00_n33_chem_13.nc',
]

# Параметры (должны совпадать с config.yaml)
VAL_SPLIT = 0.10
RANDOM_SEED = 42

# Загружаем все данные
print("Загрузка всех данных...")
all_data = load_data(ALL_FILES, base_dir='')
all_normalized = normalize_data(all_data)

print(f"Объединенные данные: {all_normalized.shape}")
print(f"Нормализованные данные: min={all_normalized.min():.4f}, max={all_normalized.max():.4f}")

print(f"\nВсего кадров: {len(all_normalized)}")

# Случайное разбиение (как при обучении!)
indices = list(range(len(all_normalized)))
train_indices, val_indices = train_test_split(
    indices, 
    test_size=VAL_SPLIT, 
    random_state=RANDOM_SEED
)

# Создаем датасеты
full_dataset = ConcentrationFieldDataset(all_normalized)
train_dataset = torch.utils.data.Subset(full_dataset, train_indices)
val_dataset = torch.utils.data.Subset(full_dataset, val_indices)

print(f"\nTrain выборка: {len(train_dataset)} кадров ({len(train_indices)/len(indices)*100:.1f}%)")
print(f"Валидационная выборка: {len(val_dataset)} кадров ({len(val_indices)/len(indices)*100:.1f}%)")
print(f"random_state={RANDOM_SEED}, val_split={VAL_SPLIT}")

# Получаем данные для обучения TT (train) и валидации (val)
train_data = all_normalized[train_indices]
val_data = all_normalized[val_indices]

print(f"\nTrain данные: {train_data.shape}")
print(f"Val данные: {val_data.shape}")

## Анализ TT с разными рангами

In [ ]:
TT_RANKS_LIST = [
    (8, 8),
    (16, 16),
    (32, 32),
    (64, 64),
    (16, 32),
    (32, 64),
    (64, 128)
]

verbose = True

print("Анализ Tensor Train с разными рангами...\n")
print("TT обучается на train данных, оценка на val данных\n")

results = {}

for tt_ranks in tqdm(TT_RANKS_LIST, desc="Анализ TT"):
    # Создаем и обучаем TT на train данных
    compressor = TTCompressor(tt_ranks=tt_ranks)
    compressor.fit(train_data, verbose=False)
    
    # Оцениваем на val данных
    compressed = compressor.transform(val_data, verbose=False)
    reconstructed = compressor.reconstruct(compressed)
    
    # Вычисляем метрики на val
    mse = np.mean((val_data - reconstructed) ** 2)
    mae = np.mean(np.abs(val_data - reconstructed))
    
    # Объяснённая дисперсия
    total_var = np.var(val_data)
    residual_var = np.var(val_data - reconstructed)
    explained_var = 1 - residual_var / total_var
    
    results[tt_ranks] = {
        'mse': mse,
        'mae': mae,
        'explained_variance': explained_var,
        'compression_ratio': compressor.compression_ratio(),
        'compressor': compressor
    }
    
    if verbose:
        print(f"\nTT-ранги={tt_ranks}:")
        print(f"  MSE: {mse:.6f}")
        print(f"  MAE: {mae:.6f}")
        print(f"  Объяснённая дисперсия: {explained_var:.4f}")
        print(f"  Коэффициент сжатия: {compressor.compression_ratio():.1f}x")

## Визуализация результатов TT

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Для визуализации используем произведение рангов как меру сложности
ranks_product = [r1 * r2 for r1, r2 in TT_RANKS_LIST]

mse_values = [results[r]['mse'] for r in TT_RANKS_LIST]
axes[0, 0].plot(ranks_product, mse_values, 'o-', linewidth=2, markersize=8)
axes[0, 0].set_xlabel('Произведение TT-рангов (r1 × r2)')
axes[0, 0].set_ylabel('MSE')
axes[0, 0].set_title('MSE ошибка реконструкции')
axes[0, 0].grid(True)
axes[0, 0].set_xscale('log')

mae_values = [results[r]['mae'] for r in TT_RANKS_LIST]
axes[0, 1].plot(ranks_product, mae_values, 'o-', linewidth=2, markersize=8, color='orange')
axes[0, 1].set_xlabel('Произведение TT-рангов (r1 × r2)')
axes[0, 1].set_ylabel('MAE')
axes[0, 1].set_title('MAE ошибка реконструкции')
axes[0, 1].grid(True)
axes[0, 1].set_xscale('log')

var_values = [results[r]['explained_variance'] for r in TT_RANKS_LIST]
axes[1, 0].plot(ranks_product, var_values, 'o-', linewidth=2, markersize=8, color='green')
axes[1, 0].set_xlabel('Произведение TT-рангов (r1 × r2)')
axes[1, 0].set_ylabel('Доля объяснённой дисперсии')
axes[1, 0].set_title('Объяснённая дисперсия')
axes[1, 0].grid(True)
axes[1, 0].set_xscale('log')
axes[1, 0].axhline(y=0.95, color='r', linestyle='--', label='95%')
axes[1, 0].axhline(y=0.99, color='orange', linestyle='--', label='99%')
axes[1, 0].legend()

compression_ratios = [results[r]['compression_ratio'] for r in TT_RANKS_LIST]
axes[1, 1].plot(ranks_product, compression_ratios, 'o-', linewidth=2, markersize=8, color='purple')
axes[1, 1].set_xlabel('Произведение TT-рангов (r1 × r2)')
axes[1, 1].set_ylabel('Коэффициент сжатия (x)')
axes[1, 1].set_title('Коэффициент сжатия')
axes[1, 1].grid(True)
axes[1, 1].set_xscale('log')

plt.tight_layout()
plt.savefig('results/plots/tt_analysis.png', dpi=150)
plt.show()

print("\n" + "="*70)
print("СВОДНАЯ ТАБЛИЦА")
print("="*70)
print(f"{'Ранги':>12} | {'r1×r2':>8} | {'MSE':>10} | {'MAE':>10} | {'Var':>8} | {'Сжатие':>8}")
print("-"*70)
for r in TT_RANKS_LIST:
    r_prod = r[0] * r[1]
    print(f"{str(r):>12} | {r_prod:>8} | {results[r]['mse']:>10.6f} | {results[r]['mae']:>10.6f} | "
          f"{results[r]['explained_variance']:>8.4f} | {results[r]['compression_ratio']:>8.1f}x")
print("="*70)

## Сравнение TT, PCA и Autoencoder

In [ ]:
MODEL_PATH = 'models/best_model.pth'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

ae_model, ae_checkpoint = load_model(MODEL_PATH, Conv3DAutoencoder, device=DEVICE)
ae_model.eval()

print(f"Автоэнкодер:")
print(f"  Best Val Loss: {ae_checkpoint.get('best_val_loss', 'N/A'):.6f}")
print(f"  Best Val MSE: {ae_checkpoint.get('best_val_mse', 'N/A'):.6f}")
print(f"  Latent dim: {ae_checkpoint.get('latent_dim', 'N/A')}")

# Берем TT с рангами (32, 32) для сравнения (примерно 64 параметра на срез)
tt_ranks = (32, 32)
tt_mse = results[tt_ranks]['mse']
tt_mae = results[tt_ranks]['mae']
tt_var = results[tt_ranks]['explained_variance']

print(f"\nTT (ранги={tt_ranks}):")
print(f"  MSE: {tt_mse:.6f}")
print(f"  MAE: {tt_mae:.6f}")
print(f"  Объяснённая дисперсия: {tt_var:.4f}")
print(f"  Коэффициент сжатия: {results[tt_ranks]['compression_ratio']:.1f}x")

In [ ]:
from torch.utils.data import DataLoader

# Используем уже готовые val_indices и val_data
n_test_samples = min(50, len(val_indices))
ae_mse_list = []
ae_mae_list = []
tt_mse_list = []
tt_mae_list = []
pca_mse_list = []
pca_mae_list = []

# Берем модели для сравнения
tt_compressor = results[(32, 32)]['compressor']

# PCA с 64 компонентами для сравнения
print("Обучение PCA для сравнения...")
pca_compressor = PCACompressor(n_components=64)
pca_compressor.fit(train_data, verbose=False)

print(f"\nТестирование на {n_test_samples} сэмплах из val выборки...")

with torch.no_grad():
    for i in range(n_test_samples):
        idx = val_indices[i]
        sample = all_normalized[idx:idx+1]
        
        # Автоэнкодер
        sample_tensor = torch.tensor(sample, dtype=torch.float32).unsqueeze(1).to(DEVICE)
        ae_rec, _ = ae_model(sample_tensor)
        ae_rec_np = ae_rec.cpu().numpy()
        
        ae_mse = np.mean((sample - ae_rec_np) ** 2)
        ae_mae = np.mean(np.abs(sample - ae_rec_np))
        ae_mse_list.append(ae_mse)
        ae_mae_list.append(ae_mae)
        
        # TT
        tt_compressed = tt_compressor.transform(sample, verbose=False)
        tt_rec = tt_compressor.reconstruct(tt_compressed)
        tt_mse_i = np.mean((sample - tt_rec) ** 2)
        tt_mae_i = np.mean(np.abs(sample - tt_rec))
        tt_mse_list.append(tt_mse_i)
        tt_mae_list.append(tt_mae_i)
        
        # PCA
        pca_compressed = pca_compressor.transform(sample)
        pca_rec = pca_compressor.reconstruct(pca_compressed)
        pca_mse_i = np.mean((sample - pca_rec) ** 2)
        pca_mae_i = np.mean(np.abs(sample - pca_rec))
        pca_mse_list.append(pca_mse_i)
        pca_mae_list.append(pca_mae_i)

print("\n" + "="*70)
print("СРАВНЕНИЕ: Autoencoder vs TT (32,32) vs PCA (64)")
print("="*70)
print(f"{'Метрика':<20} | {'Autoencoder':>15} | {'TT':>15} | {'PCA':>15}")
print("-"*70)
print(f"{'MSE (mean ± std)':<20} | {np.mean(ae_mse_list):>10.6f} ± {np.std(ae_mse_list):>4.6f} | {np.mean(tt_mse_list):>10.6f} ± {np.std(tt_mse_list):>4.6f} | {np.mean(pca_mse_list):>10.6f} ± {np.std(pca_mse_list):>4.6f}")
print(f"{'MAE (mean ± std)':<20} | {np.mean(ae_mae_list):>10.6f} ± {np.std(ae_mae_list):>4.6f} | {np.mean(tt_mae_list):>10.6f} ± {np.std(tt_mae_list):>4.6f} | {np.mean(pca_mae_list):>10.6f} ± {np.std(pca_mae_list):>4.6f}")
print("="*70)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].hist(ae_mse_list, bins=30, alpha=0.7, label=f'Autoencoder (mean={np.mean(ae_mse_list):.6f})', color='blue')
axes[0].hist(tt_mse_list, bins=30, alpha=0.7, label=f'TT (mean={np.mean(tt_mse_list):.6f})', color='green')
axes[0].hist(pca_mse_list, bins=30, alpha=0.7, label=f'PCA (mean={np.mean(pca_mse_list):.6f})', color='orange')
axes[0].set_xlabel('MSE')
axes[0].set_ylabel('Count')
axes[0].set_title('Распределение MSE')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].hist(ae_mae_list, bins=30, alpha=0.7, label=f'Autoencoder (mean={np.mean(ae_mae_list):.6f})', color='blue')
axes[1].hist(tt_mae_list, bins=30, alpha=0.7, label=f'TT (mean={np.mean(tt_mae_list):.6f})', color='green')
axes[1].hist(pca_mae_list, bins=30, alpha=0.7, label=f'PCA (mean={np.mean(pca_mae_list):.6f})', color='orange')
axes[1].set_xlabel('MAE')
axes[1].set_ylabel('Count')
axes[1].set_title('Распределение MAE')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('results/plots/ae_vs_tt_vs_pca_comparison.png', dpi=150)
plt.show()

## Визуализация примеров реконструкции

In [ ]:
# Выбираем случайные сэмплы для визуализации
sample_indices = np.random.choice(val_indices, size=3, replace=False)

fig, axes = plt.subplots(3, 4, figsize=(20, 12))

for row, idx in enumerate(sample_indices):
    sample = all_normalized[idx:idx+1]
    
    # Автоэнкодер
    with torch.no_grad():
        sample_tensor = torch.tensor(sample, dtype=torch.float32).unsqueeze(1).to(DEVICE)
        ae_rec, _ = ae_model(sample_tensor)
        ae_rec_np = ae_rec.cpu().numpy()[0, 0]
    
    # TT
    tt_compressed = tt_compressor.transform(sample, verbose=False)
    tt_rec = tt_compressor.reconstruct(tt_compressed)[0]
    
    # PCA
    pca_compressed = pca_compressor.transform(sample)
    pca_rec = pca_compressor.reconstruct(pca_compressed)[0]
    
    # Берем средний срез по высоте для визуализации
    mid_height = sample.shape[2] // 2
    
    # Оригинал
    axes[row, 0].imshow(sample[0, :, mid_height, :], cmap='viridis')
    axes[row, 0].set_title(f'Оригинал (срез h={mid_height})')
    axes[row, 0].axis('off')
    
    # Автоэнкодер
    axes[row, 1].imshow(ae_rec_np[:, mid_height, :], cmap='viridis')
    axes[row, 1].set_title(f'Autoencoder\nMSE={np.mean((sample[0] - ae_rec_np)**2):.6f}')
    axes[row, 1].axis('off')
    
    # TT
    axes[row, 2].imshow(tt_rec[:, mid_height, :], cmap='viridis')
    axes[row, 2].set_title(f'TT (32,32)\nMSE={np.mean((sample[0] - tt_rec)**2):.6f}')
    axes[row, 2].axis('off')
    
    # PCA
    axes[row, 3].imshow(pca_rec[:, mid_height, :], cmap='viridis')
    axes[row, 3].set_title(f'PCA (64)\nMSE={np.mean((sample[0] - pca_rec)**2):.6f}')
    axes[row, 3].axis('off')

plt.tight_layout()
plt.savefig('results/plots/tt_reconstruction_examples.png', dpi=150)
plt.show()

## Сохранение модели TT

In [ ]:
tt_compressor.save('models/tt_32x32.pkl')

import pickle

with open('results/tt_analysis_results.pkl', 'wb') as f:
    pickle.dump({
        'tt_ranks_list': TT_RANKS_LIST,
        'mse': mse_values,
        'mae': mae_values,
        'explained_variance': var_values,
        'results': results
    }, f)

print("\n✓ Результаты сохранены!")